In [ ]:
import esbl_parallel_with_PDH
from migen import *
import numpy as np
from matplotlib import pyplot as plt

dut = esbl_parallel_with_PDH.PhasedPurePhaseModulator()

def unpack_and_add_offsets(packed,i_of,q_of):
    i=[]
    q=[]
    for n in packed:
        buf = np.array(n,dtype=np.uint32).tobytes()
        i.append(np.frombuffer(buf[:2],dtype=np.int16)+np.array(i_of,dtype=np.int16))
        q.append(np.frombuffer(buf[2:],dtype=np.int16)+np.array(q_of,dtype=np.int16))
    return i,q


class Simulation():
    def __init__(self,f=5460000,
                      phase=0x100000,
                      i_amp=0xFFFF,
                      q_amp=0xFFFF,
                      multiplier=81,
                      phase_PFH=0,
                      size_sample=200):
        self.f = f
        self.phase = phase
        self.i_amp = i_amp
        self.q_amp = q_amp
        self.multiplier = multiplier
        self.phase_PFH = phase_PFH
        self.i_q = np.array(np.zeros(size_sample))
        self.size_sample = size_sample

    def test(self):
        yield dut.f.eq(self.f)
        yield dut.phase.eq(self.phase)
        yield dut.i_amp.eq(self.i_amp)
        yield dut.q_amp.eq(self.q_amp)
        yield dut.multiplier.eq(self.multiplier)
        yield dut.phase_PDH.eq(self.phase_PFH)
        yield

        yield dut.clr.eq(1)
        for _ in range(100):
            yield

        yield dut.clr.eq(0)
        for _ in range(16):
            yield


        for n in range(self.size_sample):
            yield
            self.i_q[n]=(yield dut.packed_output)
            
            
sim = Simulation()
    

    
run_simulation(dut,[sim.test()],vcd_name="esbl_signals.vcd")

i , q = unpack_and_add_offsets(sim.i_q,0xFFFF,0x0000)

plt.plot(i)
plt.plot(q)




